<a href="https://colab.research.google.com/github/HandikaMaulana/torrent2gdrive/blob/main/torrent2gdrive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Start Libtorrent Session

In [ ]:
!pip install libtorrent
import libtorrent as lt

settings = {'listen_interfaces': '0.0.0.0:6881'}

ses = lt.session(settings)
downloads = []

2. Import GDrive to Colab

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

3. Insert Magnet Link

In [ ]:
while True:
    magnet_link = input("Enter Magnet Link Or Type Exit: ")
    if magnet_link.lower() == "exit":
        break

    try:
        params = lt.parse_magnet_uri(magnet_link)
        params.save_path = "/content/drive/My Drive/Torrent"
        handle = ses.add_torrent(params)
        downloads.append(handle)
        print(f"Added: {handle.status().name or magnet_link}")
    except Exception as e:
        print(f"Invalid magnet link: {e}")

4. Start Downloader

In [ ]:
import time
from IPython.display import display
import ipywidgets as widgets

state_str = [
    "queued",
    "checking",
    "downloading metadata",
    "downloading",
    "finished",
    "seeding",
    "allocating",
    "checking fastresume",
]

layout = widgets.Layout(width="auto")
style = {"description_width": "initial"}
download_bars = [
    widgets.FloatSlider(
        step=0.01, disabled=True, layout=layout, style=style
    )
    for _ in downloads
]
display(*download_bars)

while downloads:
    next_shift = 0
    for index, download in enumerate(downloads[:]):
        bar = download_bars[index + next_shift]
        s = download.status()

        if not s.is_seeding:
            bar.description = " ".join(
                [
                    s.name,
                    str(s.download_rate / 1000),
                    "kB/s",
                    state_str[s.state],
                ]
            )
            bar.value = s.progress * 100
        else:
            next_shift -= 1
            ses.remove_torrent(download)
            downloads.remove(download)
            bar.close()
            download_bars.remove(bar)
            print(s.name, "complete")
    time.sleep(1)